# Fulltext Search

This notebook demonstrates how to use Neo4j fulltext indexes for keyword-based search.

Fulltext search complements vector search by enabling:
- **Exact keyword matching** - Find entities by specific terms
- **Fuzzy matching** - Handle typos and variations
- **Boolean operators** - Combine search terms with AND, OR, NOT
- **Wildcard search** - Match partial terms

---

## Setup

Import required modules and connect to Neo4j.

In [ ]:
import sys
sys.path.insert(0, '..')

from neo4j import GraphDatabase
from config import get_neo4j_driver

In [ ]:
driver = get_neo4j_driver()
driver.verify_connectivity()
print("Connected to Neo4j!")

## Verify Fulltext Index

Before running fulltext queries, verify that the required indexes exist.

In [ ]:
with driver.session() as session:
    result = session.run("""
        SHOW FULLTEXT INDEXES
        YIELD name, labelsOrTypes, properties, state
        RETURN name, labelsOrTypes, properties, state
    """)
    indexes = list(result)
    
    if indexes:
        print("Fulltext indexes found:")
        for idx in indexes:
            print(f"  - {idx['name']}: {idx['labelsOrTypes']} on {idx['properties']} ({idx['state']})")
    else:
        print("No fulltext indexes found.")
        print("You may need to create them or restore the backup.")

## Basic Fulltext Search

Search for entities by keyword using `db.index.fulltext.queryNodes`.

The procedure returns:
- `node` - The matched node
- `score` - Lucene relevance score (higher = better match)

In [ ]:
search_term = "Apple"

with driver.session() as session:
    result = session.run("""
        CALL db.index.fulltext.queryNodes('search_entities', $term)
        YIELD node, score
        RETURN labels(node) AS labels, node.name AS name, score
        LIMIT 10
    """, term=search_term)
    
    print(f"Search results for '{search_term}':")
    for record in result:
        print(f"  [{record['labels'][0]}] {record['name']} (score: {record['score']:.4f})")

## Fuzzy Search

Use the `~` operator for fuzzy matching, which handles typos and spelling variations.

In [ ]:
fuzzy_term = "Aplle~"  # Intentional typo

with driver.session() as session:
    result = session.run("""
        CALL db.index.fulltext.queryNodes('search_entities', $term)
        YIELD node, score
        RETURN labels(node) AS labels, node.name AS name, score
        LIMIT 5
    """, term=fuzzy_term)
    
    print(f"Fuzzy search results for '{fuzzy_term}':")
    for record in result:
        print(f"  [{record['labels'][0]}] {record['name']} (score: {record['score']:.4f})")

## Wildcard Search

Use `*` for multi-character wildcards and `?` for single-character wildcards.

In [ ]:
wildcard_term = "Micro*"  # Matches Microsoft, Microservices, etc.

with driver.session() as session:
    result = session.run("""
        CALL db.index.fulltext.queryNodes('search_entities', $term)
        YIELD node, score
        RETURN labels(node) AS labels, node.name AS name, score
        LIMIT 10
    """, term=wildcard_term)
    
    print(f"Wildcard search results for '{wildcard_term}':")
    for record in result:
        print(f"  [{record['labels'][0]}] {record['name']} (score: {record['score']:.4f})")

## Boolean Operators

Combine search terms using:
- `AND` - Both terms must match
- `OR` - Either term matches
- `NOT` - Exclude term

In [ ]:
boolean_term = "supply NOT chain"

with driver.session() as session:
    result = session.run("""
        CALL db.index.fulltext.queryNodes('search_entities', $term)
        YIELD node, score
        RETURN labels(node) AS labels, node.name AS name, score
        LIMIT 10
    """, term=boolean_term)
    
    print(f"Boolean search results for '{boolean_term}':")
    for record in result:
        print(f"  [{record['labels'][0]}] {record['name']} (score: {record['score']:.4f})")

## Combining Fulltext Search with Graph Traversal

The power of fulltext search in Neo4j comes from combining keyword matching with graph traversal.

In [ ]:
company_search = "Nvidia"

with driver.session() as session:
    result = session.run("""
        CALL db.index.fulltext.queryNodes('search_entities', $term)
        YIELD node, score
        WHERE 'Company' IN labels(node)
        WITH node AS company, score
        LIMIT 1
        
        // Get documents filed by company
        OPTIONAL MATCH (company)-[:FILED]->(doc:Document)
        WITH company, score, COLLECT(DISTINCT doc.path)[0..5] AS documents
        
        // Get risk factors
        OPTIONAL MATCH (company)-[:FACES_RISK]->(risk:RiskFactor)
        WITH company, score, documents, COLLECT(DISTINCT risk.name)[0..5] AS risks
        
        RETURN company.name AS company, score, documents, risks
    """, term=company_search)
    
    record = result.single()
    if record:
        print(f"Company: {record['company']} (score: {record['score']:.4f})")
        print(f"\nRelated Documents:")
        for doc in record['documents']:
            print(f"  - {doc}")
        print(f"\nRisk Factors:")
        for risk in record['risks']:
            print(f"  - {risk}")
    else:
        print(f"No company found for '{company_search}'")

## Summary

Fulltext search in Neo4j provides:

| Feature | Syntax | Example |
|---------|--------|---------|
| Basic search | `term` | `Apple` |
| Fuzzy search | `term~` | `Aplle~` |
| Wildcard | `term*` | `Micro*` |
| Boolean AND | `term1 AND term2` | `supply AND chain` |
| Boolean OR | `term1 OR term2` | `Apple OR Microsoft` |
| Boolean NOT | `term1 NOT term2` | `risk NOT financial` |
| Phrase | `"term1 term2"` | `"supply chain"` |

**When to use fulltext vs vector search:**
- **Fulltext**: Known entity names, exact terms, filtering
- **Vector**: Semantic similarity, concept matching, questions
- **Hybrid**: Combine both for best results

---

**Next:** [Hybrid Search](02_hybrid_search.ipynb)

In [ ]:
# Cleanup
driver.close()
print("Connection closed.")